# **ST 554 Homework 10**
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

In [1]:
#importing required modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sqrt

The code below creates a spark object for use with this entire assignment.

In [2]:
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 20:32:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/19 20:32:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


The code below only allows error messages to print out going forward.

In [3]:
spark.sparkContext.setLogLevel("ERROR")

## **Part 1: Creating Streaming Data Using `rate`**

The code below sets up a data stream using the `rate` format.

In [4]:
rateDF = spark.readStream.format("rate").load()

The code below sets up a sequence of actions using appropriate functions from `pyspark.sql.functions` that uses the `rate` data to:
- find the square root of the rate value
- find mod 4 of the rate value

**Note:** This is done *prior* to starting the stream!

In [5]:
manipDF = rateDF.withColumn("sqrt_value", sqrt(col("value"))) \
                .withColumn("mod4_value", col("value") % 4)

We now output the above actions by creating a `writeStream` that writes to memory. The table is named `rate_sample`, and it can be accessed with `spark.sql()`. The query is started below.

In [6]:
writeDF = manipDF.writeStream.format("memory").queryName("rate_sample").start()

After about 30 seconds, we used the code below to stop the query.

In [7]:
writeDF.stop()

The code below outputs the entire table stored in the query name.

In [9]:
spark.sql("select * from rate_sample").show(30)

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-19 20:32:...|    0|               0.0|         0|
|2026-04-19 20:32:...|    1|               1.0|         1|
|2026-04-19 20:32:...|    2|1.4142135623730951|         2|
|2026-04-19 20:32:...|    3|1.7320508075688772|         3|
|2026-04-19 20:32:...|    4|               2.0|         0|
|2026-04-19 20:32:...|    5|  2.23606797749979|         1|
|2026-04-19 20:32:...|    6| 2.449489742783178|         2|
|2026-04-19 20:32:...|    7|2.6457513110645907|         3|
|2026-04-19 20:32:...|    8|2.8284271247461903|         0|
|2026-04-19 20:32:...|    9|               3.0|         1|
|2026-04-19 20:32:...|   10|3.1622776601683795|         2|
|2026-04-19 20:32:...|   11|   3.3166247903554|         3|
|2026-04-19 20:32:...|   12|3.4641016151377544|         0|
|2026-04-19 20:32:...|   13| 3.605551275463989|         